# Data and Model Drift Concepts

---

In this notebook, we will learn about **drift**, the gradual or sudden change in data or relationships that causes a deployed model to degrade over time.

We will cover:

- What data drift and concept drift are
- Why they happen in real-world systems
- How to detect drift statistically
- When to act on drift vs. when to ignore it
- Practical monitoring strategies for freelancers

---

## 1. Types of Drift

### 1.1. Data Drift (Covariate Shift)

**Definition:** The distribution of input features changes, but the relationship between features and the target stays the same.

```
Training data:   sepal_length ~ Normal(5.8, 0.8)
Production data:  sepal_length ~ Normal(7.2, 1.5)   ← Distribution shifted
```

**Real-world examples:**  
- A pricing model trained on pre-pandemic data receives post-pandemic inputs with different spending patterns.
- A customer churn model trained on users aged 25-40 starts receiving users aged 60+ after a marketing campaign.
- A sensor calibration changes, shifting all temperatur readings by 2⁰C.

**Effect:** The model is asked to predict on inputs it has never seen. Extrapolation → unreliable predictions.

### 1.2. Concept Drift

**Definition:** The relationship between features and the target changes, even if the input distributions stays the same.

```
Training:    income > $80k → low churn risk
Production:  income > $80k → HIGH churn risk (economy changed)
```

**Real-world examples:**  
- A fraud detection model trained before a new type of fraud emerged.
- A recommendation model after a cultural shift in user preferences.
- A demand forecasting model before and after a competitor enters the market.

**Effect:** The model's learned patterns are wrong. It confidently makes incorrect predictions.

### 1.3. Comparison

|| **Data Drift** | **Concept Drift** |
| :--- | :--- | :--- |
| **What changes** | Input feature distributions | The target's relationship with features |
| **Detection** | Compare input distributions (training vs. production) | Compare model performance (accuracy over time) |
| **Can detect without labels?** | **Yes**. Only need input data. | **No**. Need ground-truth labels |
| **Fix** | Retrain on recent data | Retrain on recent data with updated labels |

---

## 2. Detecting Data Drift

Data drift is the easier type to detect because you don't need ground-truth labels - just the incoming feature values.

### 2.1. Statistical Tests

Compare the distribution of each feature between the training set and a recent window of production data:

| **Test** | **What It Compares** | **Best For** |
| :--- | :--- | :--- |
| **Kolmogorov-Smirnov (KS) Test** | Two continuous distributions | Detecting shifts in any continuous feature |
| **Chi-squared test** | Two categorical distributions | Detecting shifts in categorical features |
| **Population Stability Index (PSI)** | Binned distributions | A single score summarizing drift magnitude |

### 2.2. KS Test Example

In [ ]:
from scipy.stats import ks_2samp

# Training distribution
train_sepal_length = X_train[:, 0] # from the training set

# Recent production data (e.g., last 7 days of predictions)
prod_sepal_length = recent_inputs[:, 0]

statistic, p_value = ks_2samp(train_sepal_length, prod_sepal_length)

if p_value < 0.05:
    print(f"⚠️ Drift detected in sepal_length (p={p_value:.4f})")
else:
    print(f"✅ No significant drift in sepal_length (p={p_value:.4f})")

**Interpretation:**  
- **p-value < 0.05:** The distributions are statistically different → drift detected.
- **p-value ≥ 0.05:** No significant difference → no drift (or not enough data to detect it).

### 2.3 Population Stability Index (PSI)

PSI is a single number that quantifies how much a distribution has shifted:

In [ ]:
def calculate_psi(expected, actual, bins=10):
    """Calculate the Population Stability Index (PSI) between two distributions."""
    # Bin both distributions
    breakpoints = np.linspace(min(min(expected), min(actual)), max(max(expected), max(actual)), bins + 1)
    expected_counts = np.histogram(expected, bins=breakpoints[0] / len(expected))
    actual_counts = np.histogram(actual, bins=breakpoints[0] / len(actual))
    
    # Avoid division by zero
    expected_counts = np.clip(expected_counts, 1e-4, None)
    actual_counts = np.clip(actual_counts, 1e-4, None)
    
    psi = np.sum((actual_counts - expected_counts) * np.log(actual_counts / expected_counts))
    return psi   

| PSI Value | Interpretation |
| :--- | :--- |
| < 0.1 | No significant drift |
| 0.1 – 0.2 | Moderate drift — monitor closely |
| > 0.2 | Significant drift — investigate and likely retrain |

---

## 3. Detecting Concept Drift

Concept drift is harder to detect because you need **ground-truth labels** from production - and those are often delayed or unavailable.

### 3.1. Performance Monitoring

If you do get labels (even with a delay), track model performance over time:

In [ ]:
# Weekly accuracy over the last 8 weeks
weekly_accuracies = [0.92, 0.91, 0.89, 0.88, 0.87, 0.85, 0.84, 0.83]

A steady decline is a strong signal of concept drift.

### 3.2. Proxy Signals (When You Don't Have Labels)

If ground-truth labels aren't available, use proxy signals:

| **Signal** | **What It Tells you** |
| :--- | :--- |
| **Average prediction confidence** | If confidence drops over time, the model is less certain → possible drift. |
| **Prediction distribution** | If the model used to predict 40% class A vs. 60% class B, and now it's  10% vs 90%, something changed. |
| **Client feedback** | *"The model seems wrong more ofte."* - qualitative but valuable. |

### 3.3. Practical Approach

In [ ]:
# Track average confidence over rolling windows
daily_avg_confidence = [0.95, 0.94, 0.93, 0.92, 0.90, 0.88, 0.85, 0.80]

if daily_avg_confidence[-1] < 0.85:
    logger.warning(
        f"concept_drift_signal | avg_confidence={daily_avg_confidence[-1]:.2f} | "
        f"threshold=0.85 | action=investigate"
    )


---

## 4. When to ACT and When NOT to Act

Not every drift signal requires action. Statistical tests can be overly sensitive with large sample sizes.

### Decision Framework

```
Drift detected?
├── No → Continue monitoring
└── Yes
    ├── Is model performance actually degraded?
    │   ├── Yes → Retrain (see next notebook)
    │   └── No / Unknown
    │       ├── Is drift large? (PSI > 0.2)
    │       │   ├── Yes → Investigate and likely retrain
    │       │   └── No → Log it, increase monitoring frequency, wait
    │       └── Is confidence dropping?
    │           ├── Yes → Investigate root cause
    │           └── No → Continue monitoring
```

**The key principle:** Drift is a *signal*, not an automatic trigger. Investigate before retraining.

---

## 5. Monitoring Tools Landscape

For freelance projects, you don't need enterprise monitoring platforms. But it's good to know what exists:

| **Tool** | **Type** | **Cost** | **Notes** |
| :--- | :--- | :--- | :--- |
| **Manual logging + scripts** | DIY | Free | What we're teaching here. Good enough for most freelance projects. |
| **Evidently AI** | Open-source library | Free | Generates drift reports as HTML dashboards. Excellent for quick analysis. | 
| **Whylogs** | Open-source profiling | Free | Lightweight data profiling and drift detection. |
| **Arize AI** | SaaS platform | Paid | Full monitoring platform. Enterprise-focused. | 
| **MLflow** | Open-source platform | Free | Experiment tracking, model registry, monitoring. |

**For this learning path:** Manual loggin (previous notebook) + periodic statistical checks (this notebook) is the right level. It's practical, free, and teaches you the underlying concepts that every monitoring tool is built on.

---

## 6. Summary

| **Concept** | **Key Takeaway** |
| :--- | :--- |
| **Data drift** | Input feature distributions change. Detectable without labels (KS test, PSI). |
| **Concept drift** | The feature-target relationship changes. Requires labels or proxy signals to detect. |
| **KS test** | Compares two distributions. p < 0.05 → significant drift. |
| **PSI** | A single drift score. < 0.1 = fine, 0.1-0.2 = monitor, > 0.2 = act. |
| **Proxy signals** | Average confidence, prediction distribution, client feedback - use when labels aren't available. |
| **Drift ≠ retrain** | Drift is a signal to investigate. Only retrain when performance is actually degraded. |

---

**Next:** [Model Retraining Strategies](./03_model_retraining_strategies.ipynb): When and how to retrain a model to recover from drift.